# 📊 Donchian breakout — intraday backtest & optimize

**The idea.** Price closing beyond its highest high / lowest low of the last `entry_window` **minutes** (the classic Turtle entry, run inside the trading day) has a ~52% chance of short-term continuation — a small edge that only pays because a **tight ATR trailing stop** (`atr_mult` × ATR, trailing the best price since entry) keeps every loss small while letting real intraday trends run. Entries are blocked in the first `skip_open` / last `skip_close` minutes of each session — early bars trivially "break out" of a channel built across the overnight gap — and `eod_exit` decides whether to go flat on each day's final bar (the default, so nothing is held overnight) or carry the position into the next session — the sweep and Optuna space try both. Expect many small stop-outs and a few larger winners. Full thesis in [README.md](./README.md).

**The universe.** It trades `HighOctaneTop20`: the sibling repo's **HighOctane** universe (top volatility tercile within sector, imported with `tsim import-universes`), narrowed to the **20 most liquid names by trailing-90-day average dollar volume** (`tsim subset --universe HighOctane --top 20`) — volatility supplies breakouts, liquidity makes the 1-minute fills honest, and the pinned list in `config/universes.yaml` keeps runs reproducible.

The `data:` block deliberately loads only **~2 months of 1-minute bars** while this setup is being tested — widen `start` in [`config.yaml`](./config.yaml) once it looks sane (symbols without 1m history are skipped; backfill more via `notebooks/02_backfill.ipynb`). Every knob lives there too (defaults, sweep grid, optimization space) — this notebook only *drives* the shared engine: **load → overview → backtest → inspect → sweep → optimize → boundary-check → verify**. The same steps work for every strategy in this repo.

## 1 · Load the strategy and its data

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import warnings
warnings.filterwarnings("ignore")

from trading_simple.config import load_settings
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
print("DB:", f"{settings.db.host}:{settings.db.port}/{settings.db.name}")

from trading_simple import backtest as bt, jobs, optimize as opt, strategies
from trading_simple.data import load_panel, symbols_with_bars
from trading_simple import plots

strat = strategies.load("donchian_breakout")
print("data :", strat.data)
print("params:", strat.params)

In [ ]:
d = strat.data
syms = symbols_with_bars(engine, settings.universe_symbols(d["universe"]),
                         d["bar"], d["start"], d.get("end"), limit=d.get("max_symbols"))
panel = load_panel(engine, syms, d["bar"], d["start"], d.get("end"),
                   session=d.get("session", "regular"), exchanges=settings.exchanges)
print(f"panel: {panel.close.shape[0]:,} bars x {len(panel.symbols)} symbols "
      f"({panel.close.index[0]:%Y-%m-%d} → {panel.close.index[-1]:%Y-%m-%d})")

## 2 · Strategy overview — hyperparameters and search ranges

What each knob does, before any number crunching:

| param | role in the strategy | intuition |
|---|---|---|
| `entry_window` | channel lookback (minutes) | **the** knob: shorter = more, noisier breakouts |
| `atr_window` | ATR smoothing window (bars) | how fast the stop adapts to a name's volatility |
| `atr_mult` | trailing-stop distance in ATRs | tighter = smaller losses but more shake-outs |
| `direction` | long / short / both | shorts pay in downtrends, cost carry elsewhere |
| `skip_open` / `skip_close` | entry blackout (minutes) | no "breakouts" of the overnight gap, none into the close |
| `eod_exit` | flat at close vs hold overnight | `true` = no overnight gap risk; `false` = let winners run past the close — swept & optimized both ways |

The table below shows, per parameter: the **default** used by the single
backtest (§3), the hand-picked **sweep grid** (§5), and the **Optuna search
range** (§6). A `—` means the stage keeps the default.

In [ ]:
import pandas as pd

def _fmt_space(spec):
    """One readable line for an optuna.space entry."""
    if spec["type"] == "categorical":
        return " | ".join(map(str, spec["choices"]))
    tail = " (log)" if spec.get("log") else (f", step {spec['step']}" if spec.get("step") else "")
    return f"{spec['low']} – {spec['high']}{tail}"

pd.DataFrame({
    "default": pd.Series(strat.params),
    "sweep grid (§5)": pd.Series({k: ", ".join(map(str, v)) for k, v in strat.sweep.items()}),
    "optuna search range (§6)": pd.Series({k: _fmt_space(s)
                                           for k, s in strat.optuna["space"].items()}),
}).fillna("—")

## 3 · Single backtest with the config defaults

Engine rules (identical for every strategy): signals execute at the **next
bar's adjusted open**, realistic costs are always charged, every symbol trades
its own starting cash.

> **Capital model:** ONE shared \$10k wallet (engine cash-shared mode) — each entry deploys `cash_frac` of the *free* cash, so the book is never more than 100% invested. The repo-wide comparability rule; see `notebooks/04_compare.ipynb` for the cross-strategy scoreboard.

In [ ]:
import pandas as pd
pf = bt.run_strategy(strat, panel, settings=settings)
stats = bt.portfolio_stats(pf)
pd.Series(stats).round(4)

In [ ]:
plots.plot_equity(bt.equity_curve(pf, init_cash=settings.backtest.init_cash),
                  title=f"{strat.name} — equity, config defaults").show()

## 4 · Inspect a symbol — trades and channel overlays

`inspect_symbol` is the microscope of this notebook, used twice: here on the
most-traded symbol with the *default* parameters, and again in §10 on the
*optimized* ones. How to read the figure:

* **Price panel** — the two colored lines are the Donchian channel (the prior
  `entry_window`-minute high and low, drawn from closes as a close
  approximation). ▲ blue = long entry on an upper-band break, ▼ orange =
  short entry on a lower-band break, ✕ = exit (trailing stop hit intrabar,
  end-of-day flat, or a reversal on the opposite band); the entry→exit
  segment is green when the trade won, red when it lost.
* **Volume panel** — context only; this strategy does not read volume.

In [ ]:
def inspect_symbol(sym, pf, params):
    """Deep-dive one symbol of a portfolio: annotated chart + its trade table."""
    all_trades = bt.trade_records(pf)
    merged = {**strat.params, **params, "bar": panel.bar, "tz": panel.tz}
    overlays = (strat.indicators(panel.close[sym], panel.volume[sym], **merged)
                if strat.indicators else None)
    plots.plot_symbol(panel.close[sym], panel.volume[sym], all_trades, symbol=sym,
                      overlays=overlays, equity=pf.value(group_by=False)[sym]).show()
    mine = all_trades[all_trades["symbol"] == sym]
    return mine.sort_values("entry_ts").reset_index(drop=True).round(3)

trades = bt.trade_records(pf)
busiest = trades["symbol"].value_counts().index[0]
inspect_symbol(busiest, pf, strat.params)

## 5 · Grid sweep — a coarse map of the parameter space

Before optimizing, sweep the small hand-picked grid from `config.yaml`. The
ranked bars show the best single combos; the **heatmap grid** below them is the
multi-objective view — the same parameter pair judged by Sharpe, total return
*and* breadth (`pct_profitable` = share of symbols that made money) at once, so
a region that pays on return but is carried by one name stands out immediately.

In [ ]:
grid = bt.grid_from_config(strat.sweep, strat.valid)
print(f"{len(grid)} combos")
sweep_stats = bt.sweep(strat, panel, grid, settings=settings)
sweep_stats.sort_values("sharpe", ascending=False).head(8).round(3)

In [ ]:
plots.plot_sweep(sweep_stats).show()
plots.plot_heatmap_grid(
    sweep_stats,
    pairs=[("entry_window", "atr_mult")],
    metrics=("sharpe", "total_return", "pct_profitable"),
).show()

## 6 · Optimize (resumable, runs outside the notebook)

`jobs.run_optimize` hands the work to a `tsim optimize` subprocess. The study
scores **`excess_cagr` (maximized) against `volatility` (minimized)** — beat
buy-and-hold, as calmly as possible — so the result is a *Pareto
front* — the set of parameter sets where improving one objective must hurt the
other. Every finished trial is checkpointed to `studies/{study}.log`;
**re-running this cell resumes** exactly where it stopped (Ctrl-C / kernel
restarts lose nothing).

While it runs, every finished trial streams one line below — its objectives,
the current front size, and its parameters — so you can watch the search move.
For a richer live view, copy the **`uvx optuna-dashboard …`** command from the
first output line into a second terminal: it serves parameter importances,
objective history and the front at http://127.0.0.1:8080 while the study runs.

In [ ]:
N_TRIALS = 1000   # raise for a deeper search; re-runs accumulate into the same study
jobs.run_optimize("donchian_breakout", n_trials=N_TRIALS)

## 7 · Read the study: Pareto front and the picked trial

Gray dots are all completed trials; the blue **staircase** is the exact
boundary of the dominated region (each labeled point is one Pareto-optimal
trial); the orange star is the trial `pick_best` selects under the weights
below. Hover any point for its full parameter set.

> **Note:** new trials are floor-pruned in-objective via `optuna.min_trades` in `config.yaml`; the admissibility filter below still guards older trials in resumed journals. The study name was bumped when the capital model changed to the shared wallet — earlier journals scored independent per-symbol books and are not comparable.

In [ ]:
study = opt.load_study(strat)
front = opt.pareto_frame(study)
everything = opt.trials_frame(study)

# opt.PICK_WEIGHTS (balanced) is the repo's one preference — see optimize.py for why it
# is a constant and not a per-notebook literal. This cell used to say "full weight on
# Sharpe, half on raw return", which described the PRE-v5 objectives; the objectives are
# excess_cagr ↑ against volatility ↓ — see optimize.OBJECTIVES.
best = opt.pick_best(study)
plots.plot_pareto(front, everything, picked=best.number).show()
front.head(8).round(3)

## 8 · Boundary check — did the optimum hit the search-space edge?

A picked trial sitting **on the edge of a searched range** is a hint that the
true optimum may lie outside it. `optimize.boundary_report` compares the
picked parameters (and the whole Pareto front) with each parameter's `search:` range in
`config.yaml`: any numeric parameter within 5% of a bound — measured on the
searched scale, so `log: true` ranges are judged in log space — is flagged
with the bound to widen. Categorical parameters have no edges and are skipped.

If something is flagged: widen that bound in `config.yaml`, re-run §6 (the
same study name resumes, and the sampler explores the new region; rename the
study to restart clean), then re-read §7–§8 until the front sits inside the
box. `tsim optimize` prints the same check at the end of every CLI run.

In [ ]:
report = opt.boundary_report(best.params, strat.optuna["space"], front=front)
flagged = report[report["action"] != "ok"]
print("all clear — the searched ranges bracket the optimum" if flagged.empty
      else f"{len(flagged)} parameter(s) pressing against the search space:")
report

## 9 · Re-run the full backtest with the picked parameters

In [ ]:
print("picked params:", best.params)
pf_best = bt.run_strategy(strat, panel, best.params, settings=settings)
pd.Series(bt.portfolio_stats(pf_best)).round(4)

In [ ]:
plots.plot_equity(bt.equity_curve(pf_best, init_cash=settings.backtest.init_cash),
                  title=f"{strat.name} — equity with picked parameters").show()

## 10 · Verification — per-symbol deep dive

Portfolio means can hide an edge carried by a single name, so verify the
picked parameters symbol by symbol. First the full per-symbol breakdown of the
optimized backtest; then set `SYMBOL` in the next cell to any name that looks
interesting (the big winner, the one loser, the one with suspiciously few
trades) and re-run it to get the same annotated chart and trade table as §4 —
entries, exits, direction and the channel each breakout escaped.

In [ ]:
bt.symbol_stats(pf_best).round(4)

In [ ]:
SYMBOL = None   # ← set to any symbol from the table above (e.g. "NVDA"); None = most-traded
sym = SYMBOL or bt.trade_records(pf_best)["symbol"].value_counts().index[0]
inspect_symbol(sym, pf_best, best.params)

## Read the result honestly

* Everything above is **in-sample**: the optimizer saw all the data it is being
  judged on. Before believing a parameter set, re-run it on a *different*
  universe and a held-out date range (edit `data:` in `config.yaml`).
* Prefer combos where **`pct_profitable` is high** — an edge carried by one
  symbol is usually noise.
* More trials on the same data = more chances to overfit. The Pareto front
  going up does not mean the *strategy* got better.

**Add your own strategy:** copy `strategies/_template/`, write `signals()`, fill
in `config.yaml` — this exact notebook flow works unchanged. See the README's
"How to add a strategy".